In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import h5py
from tqdm import tqdm

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT FUNCTIONS (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from FUNCTIONS_Variable_Calculation import *

In [ ]:
#IMPORT FUNCTIONS
import sys
path=os.path.join(mainCodeDirectory,'Functions/')
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

#####

#Import StatisticalFunctions 
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
path=dir2+'Functions/'
sys.path.append(path)

import StatisticalFunctions
from StatisticalFunctions import * # import NumericalFunctions 

In [ ]:
####################################
#LOADING CLASSES

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=1)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="EntrainmentCalculation", dataName="EulerianEntrainment",
                                dtype='float32')

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=False

def GetNumJobs(res,t_res):
    if res=='1km':
        if t_res=='5min':
            num_jobs=20
        elif t_res=='3min':
            num_jobs=30
        elif t_res=='1min':
            num_jobs=100
    elif res=='250m': 
        if t_res=='1min':
            num_jobs=500
    return num_jobs
num_jobs = GetNumJobs(ModelData.res,ModelData.t_res)
SlurmJobArray = SlurmJobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = SlurmJobArray.start_job; end_job = SlurmJobArray.end_job

def GetNumElements():
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job]
    return loop_elements
loop_elements = GetNumElements()

In [ ]:
####################################
#DATA LOADING FUNCTIONS

In [ ]:
def GetVelocityData(t):
    varNames = ['winterp','vinterp','uinterp']
    velDict = {varName: CallVariable(ModelData, DataManager, ModelData.timeStrings[t], 
                                  variableName=varName)\
                     for varName in varNames}
    return velDict

def GetActiveAir(t):
    varNames = ['rho','A_g','A_c']
    activeDict = {varName: CallVariable(ModelData, DataManager,
                                                    ModelData.timeStrings[t], 
                                                    variableName=varName)\
                              for varName in varNames}
    return activeDict

In [ ]:
####################################
#COMPUTATION FUNCTIONS

In [ ]:
def ComputeEntrainment(activeDict_t1,activeDict_t0,velDict,
                       updraftType='c'):
    rhoA_t1 = activeDict_t1['rho']*activeDict_t1[f'A_{updraftType}']
    rhoA_t0 = activeDict_t0['rho']*activeDict_t0[f'A_{updraftType}']
    drhoA_dt = (rhoA_t1 - rhoA_t0) / ModelData.dt

    w = velDict['winterp']; v = velDict['vinterp']; u = velDict['uinterp']
    rhoA_mid = 0.5 * (rhoA_t0 + rhoA_t1)
    divergence = (np.gradient(rhoA_mid * w, ModelData.zh*1e3, axis=0) +
                  np.gradient(rhoA_mid * v, ModelData.yh*1e3, axis=1) +
                  np.gradient(rhoA_mid * u, ModelData.xh*1e3, axis=2))

    activitySource = drhoA_dt + divergence
    entrainment=np.maximum(0,activitySource); detrainment=np.maximum(0,-activitySource)
    return entrainment,detrainment, rhoA_mid

def DivideMassFlux(rhoA_mid_g, rhoA_mid_c, E_g, D_g, E_c, D_c, velDict):
    MF_g = np.nanmean(rhoA_mid_g * velDict['winterp'], axis=(1,2))[:, np.newaxis, np.newaxis]  
    MF_c = np.nanmean(rhoA_mid_c * velDict['winterp'], axis=(1,2))[:, np.newaxis, np.newaxis]

    E_g = NanDivide(E_g, MF_g)
    D_g = NanDivide(D_g, MF_g)
    E_c = NanDivide(E_c, MF_c)
    D_c = NanDivide(D_c, MF_c)
    
    return E_g, D_g, E_c, D_c

In [ ]:
def CalculateNET(E_g, D_g, E_c, D_c, E_g_DMF, D_g_DMF, E_c_DMF, D_c_DMF):
    NET_g     = E_g     - D_g
    NET_c     = E_c     - D_c
    NET_g_DMF = E_g_DMF - D_g_DMF
    NET_c_DMF = E_c_DMF - D_c_DMF
    return NET_g, NET_c, NET_g_DMF, NET_c_DMF

def RunCalculations(t):
    ####################################
    #DATA LOADING
    velDict = GetVelocityData(t=t)
    activeDict_t1 = GetActiveAir(t)
    activeDict_t0 = GetActiveAir(t-1)
    
    ####################################
    #COMPUTATION
    [E_g_eulerian, D_g_eulerian, rhoA_mid_g] = ComputeEntrainment(activeDict_t1, activeDict_t0, velDict, updraftType='g')
    [E_c_eulerian, D_c_eulerian, rhoA_mid_c] = ComputeEntrainment(activeDict_t1, activeDict_t0, velDict, updraftType='c')
    [E_g_eulerian_DMF, D_g_eulerian_DMF,
     E_c_eulerian_DMF, D_c_eulerian_DMF] = DivideMassFlux(rhoA_mid_g, rhoA_mid_c,
                                                          E_g_eulerian, D_g_eulerian,
                                                          E_c_eulerian, D_c_eulerian, velDict)
    # [NET_g, NET_c, NET_g_DMF, NET_c_DMF] = CalculateNET(E_g_eulerian, D_g_eulerian,
    #                                                      E_c_eulerian, D_c_eulerian,
    #                                                      E_g_eulerian_DMF, D_g_eulerian_DMF,
    #                                                      E_c_eulerian_DMF, D_c_eulerian_DMF)
    # outputDictionary = {'NET_g_eulerian': NET_g,'NET_c_eulerian': NET_c,
    #                     'NET_g_DMF_eulerian': NET_g_DMF,'NET_c_DMF_eulerian': NET_c_DMF}
    outputDictionary = {
        'E_g_eulerian':     E_g_eulerian,
        'D_g_eulerian':     D_g_eulerian,
        'E_c_eulerian':     E_c_eulerian,
        'D_c_eulerian':     D_c_eulerian,
        'E_g_eulerian_DivideMassFlux': E_g_eulerian_DMF,
        'D_g_eulerian_DivideMassFlux': D_g_eulerian_DMF,
        'E_c_eulerian_DivideMassFlux': E_c_eulerian_DMF,
        'D_c_eulerian_DivideMassFlux': D_c_eulerian_DMF,
    }
    return outputDictionary

In [ ]:
####################################
#PLOTTING FUNCTIONS

In [ ]:
def PlotEntrainment(E, D, ax1, ax2, updraftType):
    xcoord=ModelData.xh; zcoord=ModelData.zh
    
    NET = E - D
    b = np.nanmean(NET, axis=1)  # or whichever axis makes sense
    vmax = np.nanpercentile(np.abs(b), 99)
    levels = np.linspace(-vmax, vmax, 21)
    
    cf = ax1.contourf(xcoord, zcoord, b, cmap='RdBu_r', levels=levels, extend='both')
    cbar = plt.colorbar(cf, ax=ax1, 
                        label=f'E_{updraftType} - D_{updraftType} (kg m$^{-3}$ s$^{-1}$)')
    ax1.set_xlabel('x (km)'); 
    ax1.set_ylabel('z (m)'); ax1.set_ylim(0,20)
    
    ax2.plot(np.nanmean(E,   axis=(1, 2)), zcoord, 'blue',  label=f'E_{updraftType}')
    ax2.plot(np.nanmean(D,   axis=(1, 2)), zcoord, 'red',   label=f'D_{updraftType}')
    ax2.plot(np.nanmean(NET, axis=(1, 2)), zcoord, 'k',     
             label=f'E_{updraftType} - D_{updraftType}',zorder=10)
    ax2.axvline(0, color='k', lw=0.5)
    ax2.set_xlabel('kg m$^{-3}$ s$^{-1}$'); ax2.legend()
    ax2.set_ylim(0,20)

    
    apply_scientific_notation([ax1,ax2])
    apply_scientific_notation_colorbar([cbar])

def PlotEntrainmentComparison(E_eul, D_eul, E_lag, D_lag, ax1, ax2, updraftType):
    xcoord = ModelData.xh; zcoord = ModelData.zh
    
    NET_eul = E_eul - D_eul
    NET_lag = E_lag - D_lag
    
    # contourf: lagrangian - eulerian
    b = np.nanmean(NET_lag - NET_eul, axis=1)
    vmax = np.nanpercentile(np.abs(b), 99)
    levels = np.linspace(-vmax, vmax, 21)
    cf = ax1.contourf(xcoord, zcoord, b, cmap='RdBu_r', levels=levels, extend='both')
    cbar = plt.colorbar(cf, ax=ax1,
                        label=f'(E-D)$_{{{updraftType},lag}}$ - (E-D)$_{{{updraftType},eul}}$ (kg m$^{{-3}}$ s$^{{-1}}$)')
    ax1.set_xlabel('x (km)'); ax1.set_ylabel('z (m)'); ax1.set_ylim(0, 20)

    # profiles: lagrangian=solid, eulerian=dashed
    ax2.plot(np.nanmean(E_lag,   axis=(1,2)), zcoord, color='blue', ls='-',  label=f'E_{updraftType} lag')
    ax2.plot(np.nanmean(D_lag,   axis=(1,2)), zcoord, color='red',  ls='-',  label=f'D_{updraftType} lag')
    ax2.plot(np.nanmean(NET_lag, axis=(1,2)), zcoord, color='k',    ls='-',  label=f'E-D_{updraftType} lag', zorder=10)
    ax2.plot(np.nanmean(E_eul,   axis=(1,2)), zcoord, color='blue', ls='--', label=f'E_{updraftType} eul')
    ax2.plot(np.nanmean(D_eul,   axis=(1,2)), zcoord, color='red',  ls='--', label=f'D_{updraftType} eul')
    ax2.plot(np.nanmean(NET_eul, axis=(1,2)), zcoord, color='k',    ls='--', label=f'E-D_{updraftType} eul')
    ax2.axvline(0, color='k', lw=0.5)
    ax2.set_xlabel('kg m$^{-3}$ s$^{-1}$'); ax2.legend(fontsize=7, ncol=2)
    ax2.set_ylim(0, 20)

    apply_scientific_notation([ax1, ax2])
    apply_scientific_notation_colorbar([cbar])

In [ ]:
# def TestSinglePlot(t=210):
    
#     ####################################
#     #DATA LOADING
#     velDict = GetVelocityData(t=t)
#     activeDict_t1 = GetActiveAir(t)
#     activeDict_t0 = GetActiveAir(t-1)
#     lagrangEntrainDict = GetLagrangianEntrainment(t)
    
#     ####################################
#     #COMPUTATION
#     # Eulerian Entrainment
#     [E_g_eulerian,D_g_eulerian,rhoA_mid_g] = ComputeEntrainment(activeDict_t1,activeDict_t0,velDict,
#                                      updraftType='g')
#     [E_c_eulerian,D_c_eulerian,rhoA_mid_c] = ComputeEntrainment(activeDict_t1,activeDict_t0,velDict,
#                                      updraftType='c')
#     # divide by mass flux
#     [E_g_eulerian_DMF, D_g_eulerian_DMF, 
#      E_c_eulerian_DMF, D_c_eulerian_DMF] = DivideMassFlux(rhoA_mid_g, rhoA_mid_c,
#                                                           E_g_eulerian, D_g_eulerian, 
#                                                           E_c_eulerian, D_c_eulerian, 
#                                                           velDict)
    
#     ####################################
#     #PLOTTING
#     #Plotting Eulerian Entrainment
    
#     # eulerian entrainment
#     fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 10))
#     PlotEntrainment(E_g_eulerian, D_g_eulerian, ax1, ax2, updraftType='g')
#     PlotEntrainment(E_c_eulerian, D_c_eulerian, ax3, ax4, updraftType='c')
#     plt.tight_layout()

#     fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 10))
#     PlotEntrainment(E_g_eulerian_DMF, D_g_eulerian_DMF, ax1, ax2, updraftType='g')
#     PlotEntrainment(E_c_eulerian_DMF, D_c_eulerian_DMF, ax3, ax4, updraftType='c')
#     plt.tight_layout()

#     fig, axis = plt.subplots()
#     axis.plot(np.mean(rhoA_mid_g * velDict['winterp'], axis=(1,2)), ModelData.zh, label='rhoA_g*w')
#     axis.plot(np.mean(rhoA_mid_c * velDict['winterp'], axis=(1,2)), ModelData.zh, label='rhoA_c*w')
#     axis.set_xlabel('rhoAw (kg m$^{-2}$ s$^{-1}$)')
#     axis.set_ylabel('z (km)')
#     axis.legend()
#     axis.axvline(0, color='black', zorder=10)
#     axis.set_ylim(0, 20)
#     apply_scientific_notation([axis])

# TestSinglePlot(t=210)

In [ ]:
####################################
#RUNNING
running = True #keep true if running
# running = False

In [ ]:
if running:
    for t in tqdm(loop_elements):
        outputDictionary = RunCalculations(t=t)
        DataManager.SaveOutputTimestep(DataManager.outputDataDirectory, 
                                       ModelData.timeStrings[t], outputDictionary)

In [ ]:
####################################
#TESTING

In [ ]:
#testing eulerian vs lagrangian comparison

####################################
#DATA LOADING

# def ApplyLagrangianEntrainmentConstant(arr):
#     return arr * entrainmentConstant[:, np.newaxis, np.newaxis]
# entrainmentConstant = DataManager.LoadCalculations(DataManager.outputDataDirectory,
#                                                    dataName="EntrainmentConstant",
#                                                    verbose=False,)["entrainmentConstant"]
# PROCESSED_string=""
# def GetLagrangianEntrainment(t):
#     varNames = [f'{PROCESSED_string}Entrainment_g',f'{PROCESSED_string}Detrainment_g',f'{PROCESSED_string}Entrainment_c',f'{PROCESSED_string}Detrainment_c']
#     lagrangEntrainDict = {varName: CallVariable(ModelData, DataManager,
#                                                 ModelData.timeStrings[t], 
#                                                 variableName=varName)\
#                           for varName in varNames}
#     lagrangEntrainDict = {varName: ApplyLagrangianEntrainmentConstant(lagrangEntrainDict[varName]) for varName in varNames}
#     return lagrangEntrainDict


####################################
#COMPUTATION

# # Lagrangian Entrainment
# E_g_lagrangian = lagrangEntrainDict[f'{PROCESSED_string}Entrainment_g']
# D_g_lagrangian = -lagrangEntrainDict[f'{PROCESSED_string}Detrainment_g']
# E_c_lagrangian = lagrangEntrainDict[f'{PROCESSED_string}Entrainment_c']
# D_c_lagrangian = -lagrangEntrainDict[f'{PROCESSED_string}Detrainment_c']

####################################
#PLOTTING

# #Comparing Eulerian and Lagrangian Entrainment

# # eulerian entrainment
# fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 10))
# PlotEntrainment(E_g_eulerian, D_g_eulerian, ax1, ax2, updraftType='g')
# PlotEntrainment(E_c_eulerian, D_c_eulerian, ax3, ax4, updraftType='c')
# plt.tight_layout()

# # lagrangian entrainment
# fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 10))
# PlotEntrainment(E_g_lagrangian, D_g_lagrangian, ax1, ax2, updraftType='g')
# PlotEntrainment(E_c_lagrangian, D_c_lagrangian, ax3, ax4, updraftType='c')
# plt.tight_layout()

# #lagrangian vs eulerian entrainment
# fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 10))
# PlotEntrainmentComparison(E_g_eulerian, D_g_eulerian, E_g_lagrangian, D_g_lagrangian, ax1, ax2, updraftType='g')
# PlotEntrainmentComparison(E_c_eulerian, D_c_eulerian, E_c_lagrangian, D_c_lagrangian, ax3, ax4, updraftType='c')
# plt.tight_layout()